# Space Invader mosaic — DINOv2 metric learning

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `colab_training_data.zip` to your Google Drive root
3. Run all cells in order

Training mixes two triplet types:
- **Reference triplets** — augmented reference crop → clean reference crop → different crop
- **Flash triplets** — real labeled flash image → reference crop of correct invader → different crop

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
zip_path     = '/content/drive/MyDrive/colab_training_data.zip'
extract_path = '/content/data'

if not os.path.exists(extract_path):
    print('Unzipping...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_path)
    print('Done')
else:
    print('Already extracted')

Mounted at /content/drive
Unzipping...
Done


In [2]:
!pip install -q transformers albumentations ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.8 MB/s eta 0:00:00


In [3]:
import json, random, time
from pathlib import Path

import albumentations as A
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel

# ── Configuration ────────────────────────────────────────────────
IMAGE_ROOT      = Path('/content/data/images')
FLASH_ROOT      = Path('/content/data/flash_images')
MANIFEST        = Path('/content/data/reference_manifest.jsonl')
FLASH_LABELS    = Path('/content/data/flash_labels.jsonl')
DETECTOR_PATH   = Path('/content/data/mosaic_detector_v3.pt')
OUTPUT_DIR      = Path('/content/drive/MyDrive/dinov2_finetuned_aug_crop_labeled')
BASE_MODEL      = 'facebook/dinov2-small'
TOTAL_EPOCHS    = 40
START_EPOCH     = 1
END_EPOCH       = 40
BATCH_SIZE      = 32
LR              = 1e-5
MARGIN          = 0.3
UNFREEZE_N      = 2
FLASH_OVERSAMPLE = 20   # each labeled flash image generates this many triplets per epoch
# ─────────────────────────────────────────────────────────────────

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Running epochs {START_EPOCH} – {END_EPOCH} of {TOTAL_EPOCHS}')

Device: cuda
Running epochs 1 – 20 of 20


In [4]:
FLASH_AUG = A.Compose([
    A.ColorJitter(brightness=0.6, contrast=0.6, saturation=0.4, hue=0.1, p=0.9),
    A.GaussianBlur(blur_limit=(3, 7), sigma_limit=(0.5, 3.0), p=0.6),
    A.Perspective(scale=(0.05, 0.15), p=0.5),
    A.Affine(rotate=(-20, 20), translate_percent=(-0.1, 0.1), scale=(0.75, 1.25), p=0.6),
    A.RandomBrightnessContrast(p=0.4),
    A.Sharpen(alpha=(0, 0.5), lightness=(0.5, 1.0), p=0.3),
    A.ToGray(p=0.05),
    A.ImageCompression(quality_range=(60, 95), p=0.3),
])

LIGHT_AUG = A.Compose([
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

def open_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert('RGB')


class MosaicDetector:
    def __init__(self, model_path, conf=0.25):
        from ultralytics import YOLO
        self._model = YOLO(str(model_path))
        self._conf  = conf

    def crop(self, image_path):
        img = open_image(image_path)
        results = self._model(img, conf=self._conf, verbose=False)
        boxes = results[0].boxes
        if boxes is None or len(boxes) == 0:
            return None
        best = int(boxes.conf.argmax())
        x1, y1, x2, y2 = boxes.xyxy[best].tolist()
        return img.crop((x1, y1, x2, y2))


class TripletDataset(Dataset):
    """Mixes reference triplets and labeled flash triplets.

    Reference triplet: augmented ref crop → clean ref crop → different ref crop
    Flash triplet:     real flash crop    → ref crop of correct invader → different ref crop
    """
    def __init__(self, by_id, processor, triplets_per_invader=4, augment=False,
                 detector=None, flash_pairs=None, flash_oversample=20):
        self._by_id     = {k: v for k, v in by_id.items() if len(v) >= 2}
        self._ids       = list(self._by_id.keys())
        self._processor = processor
        self._augment   = augment
        self._tpi       = triplets_per_invader
        self._ref_len   = len(self._ids) * triplets_per_invader
        # Flash pairs: list of (flash_image_path, invader_id)
        self._flash_pairs    = flash_pairs or []
        self._flash_oversample = flash_oversample
        self._flash_len      = len(self._flash_pairs) * flash_oversample

        # Pre-compute detector crops for all reference images
        self._ref_crops = {}
        if detector is not None:
            all_paths = {p for paths in self._by_id.values() for p in paths}
            print(f'Pre-computing reference crops for {len(all_paths)} images...')
            for i, path in enumerate(sorted(all_paths), 1):
                if i % 200 == 0 or i == len(all_paths):
                    print(f'  {i}/{len(all_paths)}', end='\r')
                self._ref_crops[path] = detector.crop(IMAGE_ROOT / path)
            found = sum(1 for v in self._ref_crops.values() if v is not None)
            print(f'  Reference crops: {found}/{len(all_paths)} ({100*found//len(all_paths)}%)')

        # Pre-compute detector crops for labeled flash images
        self._flash_crops = {}
        if detector is not None and self._flash_pairs:
            print(f'Pre-computing flash crops for {len(self._flash_pairs)} labeled images...')
            for path, _ in self._flash_pairs:
                self._flash_crops[path] = detector.crop(path)
            found = sum(1 for v in self._flash_crops.values() if v is not None)
            print(f'  Flash crops: {found}/{len(self._flash_pairs)} ({100*found//len(self._flash_pairs)}%)')

    def __len__(self):
        return self._ref_len + self._flash_len

    def _load_ref(self, path, aug=None):
        img = self._ref_crops.get(path)
        if img is None:
            img = open_image(IMAGE_ROOT / path)
        if aug is not None:
            img = aug(image=np.array(img))['image']
        return self._processor(images=img, return_tensors='pt')['pixel_values'].squeeze(0)

    def _load_flash(self, path, aug=None):
        img = self._flash_crops.get(path)
        if img is None:
            img = open_image(path)
        if aug is not None:
            img = aug(image=np.array(img))['image']
        return self._processor(images=img, return_tensors='pt')['pixel_values'].squeeze(0)

    def __getitem__(self, idx):
        if idx < self._ref_len:
            # Reference triplet
            anchor_id = self._ids[idx % len(self._ids)]
            anchor_path, pos_path = random.sample(self._by_id[anchor_id], 2)
            neg_id   = random.choice([i for i in self._ids if i != anchor_id])
            neg_path = random.choice(self._by_id[neg_id])
            if self._augment:
                return self._load_ref(anchor_path, FLASH_AUG), self._load_ref(pos_path, LIGHT_AUG), self._load_ref(neg_path)
            return self._load_ref(anchor_path), self._load_ref(pos_path), self._load_ref(neg_path)
        else:
            # Flash triplet: real flash → correct ref → wrong ref
            flash_path, invader_id = self._flash_pairs[(idx - self._ref_len) % len(self._flash_pairs)]
            pos_options = self._by_id.get(invader_id)
            if not pos_options:
                # Invader not in train set — fall back to ref triplet
                return self.__getitem__(idx % self._ref_len)
            pos_path = random.choice(pos_options)
            neg_id   = random.choice([i for i in self._ids if i != invader_id])
            neg_path = random.choice(self._by_id[neg_id])
            # Flash anchor gets light aug only — it's already a real flash photo
            return self._load_flash(flash_path, LIGHT_AUG), self._load_ref(pos_path), self._load_ref(neg_path)


def encode(model, pixel_values):
    out = model(pixel_values=pixel_values)
    return F.normalize(out.last_hidden_state[:, 0], dim=-1)

def split_by_invader(rows, val_fraction=0.15):
    by_id = {}
    for r in rows:
        by_id.setdefault(r['invader_id'], []).append(r['image_path'])
    trainable = [k for k, v in by_id.items() if len(v) >= 2]
    random.seed(42)
    random.shuffle(trainable)
    n_val   = max(1, int(len(trainable) * val_fraction))
    val_ids = set(trainable[:n_val])
    train   = {k: v for k, v in by_id.items() if k not in val_ids and len(v) >= 2}
    val     = {k: v for k, v in by_id.items() if k in val_ids}
    return train, val

print('Code loaded')

Code loaded


In [5]:
rows = [json.loads(l) for l in MANIFEST.read_text().splitlines() if l.strip()]
train_inv, val_inv = split_by_invader(rows)
print(f'Train invaders: {len(train_inv)}  Val invaders: {len(val_inv)}')

# Load labeled flash pairs
flash_pairs = []
if FLASH_LABELS.exists():
    flash_rows = [json.loads(l) for l in FLASH_LABELS.read_text().splitlines() if l.strip()]
    for r in flash_rows:
        path = FLASH_ROOT / f"flash/{Path(r['image_path']).name}"
        if path.exists() and r['invader_id'] in train_inv:
            flash_pairs.append((str(path), r['invader_id']))
    print(f'Labeled flash pairs available for training: {len(flash_pairs)}')
    skipped = len(flash_rows) - len(flash_pairs)
    if skipped:
        print(f'  ({skipped} skipped — image missing or invader in val set)')
else:
    print('No flash_labels.jsonl found')

# Load detector
detector = None
if DETECTOR_PATH.exists():
    detector = MosaicDetector(DETECTOR_PATH)
    print('Detector loaded')
else:
    print('No detector weights found — using full images')

# Resume from Drive checkpoint if continuing, otherwise start from base model
state_path   = OUTPUT_DIR / 'training_state.json'
resume       = START_EPOCH > 1 and OUTPUT_DIR.exists()
model_source = str(OUTPUT_DIR) if resume else BASE_MODEL
print(f'Loading model from: {model_source}')

processor = AutoImageProcessor.from_pretrained(model_source)
model     = AutoModel.from_pretrained(model_source)

for p in model.parameters():
    p.requires_grad = False
for block in model.encoder.layer[-UNFREEZE_N:]:
    for p in block.parameters():
        p.requires_grad = True
for p in model.layernorm.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}')
model.to(device)

train_ds = TripletDataset(
    train_inv, processor, augment=True, detector=detector,
    flash_pairs=flash_pairs, flash_oversample=FLASH_OVERSAMPLE
)
val_ds = TripletDataset(val_inv, processor, triplets_per_invader=8, augment=False, detector=detector)
print(f'Train triplets per epoch: {len(train_ds)} ({train_ds._ref_len} ref + {train_ds._flash_len} flash)')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, last_epoch=START_EPOCH - 2)

best_val_loss = float('inf')
if resume and state_path.exists():
    state = json.loads(state_path.read_text())
    best_val_loss = state.get('best_val_loss', float('inf'))
    print(f'Resumed — best val loss so far: {best_val_loss:.4f}')

print('Ready to train')

Train invaders: 1734  Val invaders: 305
Labeled flash pairs available for training: 319
  (94 skipped — image missing or invader in val set)
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Detector loaded
Loading model from: facebook/dinov2-small


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Trainable: 3,551,232 / 22,056,576
Pre-computing reference crops for 3942 images...
  Reference crops: 3119/3942 (79%)
Pre-computing flash crops for 319 labeled images...
  Flash crops: 319/319 (100%)
Pre-computing reference crops for 695 images...
  Reference crops: 535/695 (76%)
Train triplets per epoch: 13316 (6936 ref + 6380 flash)
Ready to train


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(START_EPOCH, END_EPOCH + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0
    for anchor, positive, negative in train_loader:
        pv  = torch.cat([anchor, positive, negative], dim=0).to(device)
        emb = encode(model, pv)
        b   = anchor.shape[0]
        loss = F.triplet_margin_loss(emb[:b], emb[b:2*b], emb[2*b:], margin=MARGIN)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for anchor, positive, negative in val_loader:
            pv  = torch.cat([anchor, positive, negative], dim=0).to(device)
            emb = encode(model, pv)
            b   = anchor.shape[0]
            val_loss += F.triplet_margin_loss(emb[:b], emb[b:2*b], emb[2*b:], margin=MARGIN).item()
    val_loss /= len(val_loader)
    scheduler.step()

    elapsed = time.time() - t0
    print(f'Epoch {epoch:3d}/{TOTAL_EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  {elapsed:.0f}s')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        state_path.write_text(json.dumps({'best_val_loss': best_val_loss, 'epoch': epoch}))
        print(f'  -> saved to Drive (best so far)')

print(f'\nDone. Best val loss: {best_val_loss:.4f}')
print(f'Weights at: {OUTPUT_DIR}')

Epoch   1/20  train=0.1374  val=0.1792  418s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> saved to Drive (best so far)
Epoch   2/20  train=0.0911  val=0.1649  417s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> saved to Drive (best so far)
Epoch   3/20  train=0.0741  val=0.1575  409s


...

  -> saved to Drive (best so far)
Epoch  15/20  train=0.0270  val=0.1325  406s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> saved to Drive (best so far)
Epoch  16/20  train=0.0268  val=0.1350  400s
Epoch  17/20  train=0.0277  val=0.1350  398s
Epoch  18/20  train=0.0269  val=0.1331  404s
Epoch  19/20  train=0.0266  val=0.1287  404s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> saved to Drive (best so far)
Epoch  20/20  train=0.0252  val=0.1365  405s

Done. Best val loss: 0.1287
Weights at: /content/drive/MyDrive/dinov2_finetuned_aug_crop_labeled
